# Notebook final — Figuras cualitativas BAP2 CHILD-ONLY

Objetivo: generar figuras cualitativas donde **todas las imágenes generadas sean de niños/adolescentes**, no adultos.

Figuras:

1. **Figura 1 — Comparación a edad infantil real**
   ```text
   Father | Mother | Real Child | Ours
   ```
   Solo selecciona tríadas donde la edad real del hijo es menor o igual a `MAX_CHILD_REAL_AGE`.

2. **Figura 2 — Progresión infantil**
   Sin GPU:
   ```text
   Father | Mother | Real Child | Age 5 | Age 15
   ```
   porque esas son las edades infantiles ya disponibles en tu test final.

   Con GPU opcional:
   ```text
   Father | Mother | Real Child | Age 5 | Age 8 | Age 10 | Age 12 | Age 15
   ```
   generando pocas imágenes nuevas.

3. **Figura 3 — Limitaciones infantiles**
   Solo casos en edad 5 o 15.

Recomendación inicial: ejecutar con `GENERATE_CHILD_GRID = False` para no gastar GPU.  
Si la figura queda pobre, activar `GENERATE_CHILD_GRID = True` y generar solo 4–6 tríadas × 5 edades infantiles.

## Inicio comun


In [ ]:
# === Arranque comun del proyecto ===
# Bootstrap Colab/Drive + src path
import os, sys

DRIVE = "/content/drive/MyDrive/Notebook4"
try:
    from google.colab import drive
    drive.mount("/content/drive")
except Exception:
    pass

for _path in [DRIVE, "/content", "/content/IP-Adapter"]:
    if _path not in sys.path:
        sys.path.insert(0, _path)


from src.config import DEFAULT_DRIVE
from src.inference_utils import (
    CHILD_NEGATIVE_PROMPT,
    DEFAULT_NEGATIVE_PROMPT,
    collect_bap2_alpha_stats,
    ensure_2d,
    generate_bap2_image,
    get_embedding,
    set_parent_scale_bap2,
)

from src.notebook_setup import setup_runtime

DRIVE = globals().get("DRIVE", DEFAULT_DRIVE)
ctx = setup_runtime(DRIVE, require_cuda=False)
paths = ctx.paths
runtime = ctx.runtime

DEVICE = runtime.device
DTYPE = runtime.dtype
FIW_ROOT = str(paths.fiw_root)
FINAL_TEST_DIR = f"{DRIVE}/results/eval"
BAP2_CSV = f"{FINAL_TEST_DIR}/bap2/bap2.csv"
OUT_DIR = f"{DRIVE}/results/figs_child"

print('Contexto base CHILD-ONLY listo')
print('Para generar con GPU, usa setup_generation_models(ctx) y setup_final_bap2(ctx).')


In [5]:
# ============================================================
# 1. Configuración CHILD-ONLY
# ============================================================

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from PIL import Image, ImageOps
from IPython.display import display

# ------------------------------------------------------------
# Modo infantil
# ------------------------------------------------------------

# Edades infantiles ya disponibles en tu test final.
EXISTING_CHILD_AGES = [5, 15]

# Edades infantiles extra si decides gastar GPU para una progresión más bonita.
# Si GENERATE_CHILD_GRID=False, esta lista NO se usa para generar.
CHILD_AGE_GRID_TO_GENERATE = [5, 8, 10, 12, 15]

# Límite para seleccionar hijos reales infantiles/adolescentes.
MAX_CHILD_REAL_AGE = 16

# Figuras
N_REALAGE_ROWS = 8
N_PROGRESSION_ROWS = 5
N_LIMITATION_ROWS = 4

# GPU opcional:
# False = no genera nada nuevo, usa solo edades 5 y 15 ya existentes.
# True  = genera pocas imágenes nuevas para CHILD_AGE_GRID_TO_GENERATE.
GENERATE_CHILD_GRID = True

# Si GENERATE_CHILD_GRID=True, también puede generar imagen a edad real redondeada.
GENERATE_REAL_CHILD_AGE_EXACT = True

# Parámetros visuales
IMG_SIZE = 230
DPI = 300

# Selección manual opcional. Deja vacío para selección automática.
MANUAL_REALAGE_TRIADS = []
MANUAL_PROGRESSION_TRIADS = []

print("GENERATE_CHILD_GRID:", GENERATE_CHILD_GRID)
print("GENERATE_REAL_CHILD_AGE_EXACT:", GENERATE_REAL_CHILD_AGE_EXACT)
print("MAX_CHILD_REAL_AGE:", MAX_CHILD_REAL_AGE)
print("EXISTING_CHILD_AGES:", EXISTING_CHILD_AGES)
print("CHILD_AGE_GRID_TO_GENERATE:", CHILD_AGE_GRID_TO_GENERATE)
print("OUT_DIR:", OUT_DIR)

GENERATE_CHILD_GRID: True
GENERATE_REAL_CHILD_AGE_EXACT: True
MAX_CHILD_REAL_AGE: 16
EXISTING_CHILD_AGES: [5, 15]
CHILD_AGE_GRID_TO_GENERATE: [5, 8, 10, 12, 15]
OUT_DIR: /content/drive/MyDrive/Notebook4/results/figs_child


In [6]:
# ============================================================
# 2. Cargar resultados finales y edades
# ============================================================

bap2_df = pd.read_csv(BAP2_CSV)
ages_df = pd.read_csv(AGES_CSV)

print("bap2_df:", bap2_df.shape)
print("ages_df:", ages_df.shape)
print("BAP2 parent_scale:", sorted(bap2_df["parent_scale"].unique()))
print("BAP2 edades disponibles:", sorted(bap2_df["age"].unique()))
print("BAP2 tríadas:", bap2_df["triad_idx"].nunique())

display(bap2_df.head(2))
display(ages_df.head(2))

bap2_df: (2080, 21)
ages_df: (1967, 2)
BAP2 parent_scale: [np.float64(0.0), np.float64(0.1)]
BAP2 edades disponibles: [np.int64(5), np.int64(15), np.int64(25), np.int64(40), np.int64(60)]
BAP2 tríadas: 208


,variant,checkpoint_step,triad_idx,age,parent_scale,gender,S_id_child,S_p_father,S_m_mother,S_family,...,alpha_mean,alpha_std,alpha_min,alpha_max,key_p,key_m,key_c,seed,prompt,image_path
0,BAP2_step4689,4689,0,5,0.0,woman,0.034026,-0.014009,0.037004,0.011498,...,0.497972,0.026212,0.373463,0.609387,F0003/MID2/P00019_face2.jpg,F0003/MID3/P00022_face1.jpg,F0003/MID1/P00019_face1.jpg,47,"portrait photo of a woman, age 5, realistic, n...",/content/drive/MyDrive/Notebook4/results/final...
1,BAP2_step4689,4689,0,5,0.1,woman,0.064559,-0.008916,0.015542,0.003313,...,0.497972,0.026263,0.373189,0.609731,F0003/MID2/P00019_face2.jpg,F0003/MID3/P00022_face1.jpg,F0003/MID1/P00019_face1.jpg,47,"portrait photo of a woman, age 5, realistic, n...",/content/drive/MyDrive/Notebook4/results/final...


,key,age
0,F0422/MID1/P04434_face4.jpg,34
1,F0184/MID4/P01972_face1.jpg,15


In [7]:
# ============================================================
# 3. Detectar columnas de edad y construir edad real del hijo
# ============================================================

def guess_col(cols, candidates):
    low = {str(c).lower(): c for c in cols}
    for cand in candidates:
        cand = cand.lower()
        for lc, orig in low.items():
            if cand in lc:
                return orig
    return None

age_key_col = guess_col(ages_df.columns, ["key", "path", "image", "file", "filename", "img"])
age_val_col = guess_col(ages_df.columns, ["age", "edad"])

print("age_key_col:", age_key_col)
print("age_val_col:", age_val_col)

assert age_key_col is not None, "No encontré columna key/path en mivolo_ages.csv"
assert age_val_col is not None, "No encontré columna de edad en mivolo_ages.csv"

tmp_age = ages_df[[age_key_col, age_val_col]].copy()
tmp_age[age_key_col] = tmp_age[age_key_col].astype(str)
tmp_age[age_val_col] = pd.to_numeric(tmp_age[age_val_col], errors="coerce")

age_map = dict(zip(tmp_age[age_key_col], tmp_age[age_val_col]))

def lookup_age_for_key(key):
    key = str(key)
    if key in age_map:
        return age_map[key]
    matches = [v for k, v in age_map.items() if str(k).endswith(key)]
    if len(matches) > 0:
        return matches[0]
    return np.nan

def nearest_age(age, grid):
    if pd.isna(age):
        return grid[0]
    age = float(age)
    return min(grid, key=lambda x: abs(float(x) - age))

age_key_col: key
age_val_col: age


In [8]:
# ============================================================
# 4. Construir resumen por tríada BAP2 λ=0.1
# ============================================================

bap2_on = bap2_df[bap2_df["parent_scale"] == 0.1].copy()

triad_summary = (
    bap2_on
    .groupby("triad_idx")
    .agg(
        n_ages=("age", "nunique"),
        mean_S_id=("S_id_child", "mean"),
        mean_S_family=("S_family", "mean"),
        mean_gap=("balance_abs_gap", "mean"),
        key_p=("key_p", "first"),
        key_m=("key_m", "first"),
        key_c=("key_c", "first"),
        gender=("gender", "first"),
    )
    .reset_index()
)

triad_summary["real_age_child"] = triad_summary["key_c"].apply(lookup_age_for_key)
triad_summary["nearest_existing_child_age"] = triad_summary["real_age_child"].apply(
    lambda x: nearest_age(x, EXISTING_CHILD_AGES)
)

print("triad_summary:", triad_summary.shape)
print("Edades reales faltantes:", triad_summary["real_age_child"].isna().sum())

print("Tríadas con hijo real <= MAX_CHILD_REAL_AGE:")
print(((triad_summary["real_age_child"].notna()) & (triad_summary["real_age_child"] <= MAX_CHILD_REAL_AGE)).sum())

display(triad_summary.head())

triad_summary: (208, 11)
Edades reales faltantes: 0
Tríadas con hijo real <= MAX_CHILD_REAL_AGE:
95


,triad_idx,n_ages,mean_S_id,mean_S_family,mean_gap,key_p,key_m,key_c,gender,real_age_child,nearest_existing_child_age
0,0,5,0.118379,-0.016580,0.027535,F0003/MID2/P00019_face2.jpg,F0003/MID3/P00022_face1.jpg,F0003/MID1/P00019_face1.jpg,woman,43,15
1,1,5,0.151839,0.007896,0.066483,F0003/MID2/P00019_face2.jpg,F0003/MID3/P00022_face1.jpg,F0003/MID4/P00023_face1.jpg,woman,49,15
2,2,5,0.372233,0.064556,0.054859,F0008/MID1/P00083_face2.jpg,F0008/MID7/P00091_face1.jpg,F0008/MID2/P00083_face1.jpg,woman,8,5
3,3,5,0.194586,0.041308,0.034584,F0008/MID1/P00083_face2.jpg,F0008/MID7/P00091_face1.jpg,F0008/MID3/P00083_face3.jpg,woman,7,5
4,4,5,0.253781,0.030579,0.059745,F0008/MID4/P00084_face1.jpg,F0008/MID9/P00095_face2.jpg,F0008/MID8/P00094_face2.jpg,man,6,5


In [9]:
# ============================================================
# 5. Helpers de imagen
# ============================================================

from src.image_utils import (
    image_quality_stats,
    key_to_path as _key_to_path,
    prep_img as _prep_img,
    read_img,
    square_center_crop,
)

def key_to_path(key):
    return str(_key_to_path(FIW_ROOT, key))

def prep_img(path, size=IMG_SIZE):
    return _prep_img(path, size=size)

def save_fig(fig, png_path, pdf_path=None):
    fig.savefig(png_path, dpi=DPI, bbox_inches="tight", facecolor="white")
    if pdf_path is not None:
        fig.savefig(pdf_path, bbox_inches="tight", facecolor="white")
    print("Guardado:", png_path)
    if pdf_path:
        print("Guardado:", pdf_path)

def get_existing_bap2_image_path(triad_idx, age):
    sub = bap2_on[
        (bap2_on["triad_idx"] == int(triad_idx)) &
        (bap2_on["age"] == int(age))
    ]
    if len(sub) != 1:
        raise ValueError(f"No encontre imagen BAP2 triad_idx={triad_idx}, age={age}")
    return sub.iloc[0]["image_path"]


In [10]:
# ============================================================
# 6. Score visual + filtro CHILD-ONLY
# ============================================================

vis = triad_summary.copy()
vis["father_path"] = vis["key_p"].apply(key_to_path)
vis["mother_path"] = vis["key_m"].apply(key_to_path)
vis["child_path"]  = vis["key_c"].apply(key_to_path)

for prefix, col in [("f", "father_path"), ("m", "mother_path"), ("c", "child_path")]:
    stats = vis[col].apply(image_quality_stats).apply(pd.Series).add_prefix(prefix + "_")
    vis = pd.concat([vis, stats], axis=1)

vis["real_min_side"] = vis[["f_min_side", "m_min_side", "c_min_side"]].min(axis=1)
vis["real_sharpness_mean"] = vis[["f_sharpness", "m_sharpness", "c_sharpness"]].mean(axis=1)
vis["real_contrast_mean"] = vis[["f_contrast", "m_contrast", "c_contrast"]].mean(axis=1)
vis["real_colorfulness_mean"] = vis[["f_colorfulness", "m_colorfulness", "c_colorfulness"]].mean(axis=1)
vis["has_gray_like_real"] = (
    vis["f_gray_like"].fillna(True) |
    vis["m_gray_like"].fillna(True) |
    vis["c_gray_like"].fillna(True)
)

vis["rank_id"] = vis["mean_S_id"].rank(pct=True)
vis["rank_family"] = vis["mean_S_family"].rank(pct=True)
vis["rank_sharp"] = vis["real_sharpness_mean"].rank(pct=True)
vis["rank_contrast"] = vis["real_contrast_mean"].rank(pct=True)
vis["rank_color"] = vis["real_colorfulness_mean"].rank(pct=True)

vis["figure_score"] = (
    0.30 * vis["rank_id"] +
    0.25 * vis["rank_family"] +
    0.20 * vis["rank_sharp"] +
    0.15 * vis["rank_contrast"] +
    0.10 * vis["rank_color"]
)

# CHILD-ONLY: hijos reales <= MAX_CHILD_REAL_AGE
child_pool = vis[
    (vis["real_age_child"].notna()) &
    (vis["real_age_child"] <= MAX_CHILD_REAL_AGE) &
    (vis["real_age_child"] >= 0) &
    (vis["real_min_side"] >= 80)
].copy()

print("child_pool:", child_pool.shape)

if len(child_pool) < max(N_REALAGE_ROWS, N_PROGRESSION_ROWS):
    print("⚠ Pocos niños reales. Relajando calidad visual, pero manteniendo edad infantil.")
    child_pool = vis[
        (vis["real_age_child"].notna()) &
        (vis["real_age_child"] <= MAX_CHILD_REAL_AGE) &
        (vis["real_age_child"] >= 0)
    ].copy()

assert len(child_pool) > 0, "No hay tríadas con edad infantil. Revisa MAX_CHILD_REAL_AGE o ages_csv."

display(child_pool[[
    "triad_idx", "real_age_child", "nearest_existing_child_age",
    "mean_S_id", "mean_S_family", "real_min_side", "figure_score"
]].sort_values("figure_score", ascending=False).head(15))

child_pool: (95, 40)


,triad_idx,real_age_child,nearest_existing_child_age,mean_S_id,mean_S_family,real_min_side,figure_score
53,53,9,5,0.322190,0.092529,108,0.850962
130,130,10,5,0.286577,0.028453,108,0.795673
126,126,5,5,0.319355,0.040284,108,0.776442
149,149,12,15,0.242055,0.034242,108,0.740385
203,203,12,15,0.233171,0.044482,108,0.736779
186,186,6,5,0.316862,0.051410,108,0.734856
127,127,2,5,0.232044,0.048482,108,0.734375
184,184,10,5,0.400207,0.030100,108,0.714663
174,174,5,5,0.173068,0.031583,108,0.710337
109,109,2,5,0.212748,0.009321,108,0.705529


In [23]:
# ============================================================
# 7. Selección de tríadas CHILD-ONLY
# ============================================================

pool_sorted = child_pool.sort_values("figure_score", ascending=False).reset_index(drop=True)

if len(MANUAL_REALAGE_TRIADS) > 0:
    selected_realage = child_pool[child_pool["triad_idx"].isin(MANUAL_REALAGE_TRIADS)].copy()
else:
    selected_realage = pool_sorted.head(N_REALAGE_ROWS).copy()

if len(MANUAL_PROGRESSION_TRIADS) > 0:
    selected_progression = child_pool[child_pool["triad_idx"].isin(MANUAL_PROGRESSION_TRIADS)].copy()
else:
    # Elegir progresión de niños distribuidos dentro de los mejores.
    qs = np.linspace(0.00, 0.35, N_PROGRESSION_ROWS)
    idxs = [min(int(q * (len(pool_sorted) - 1)), len(pool_sorted) - 1) for q in qs]
    selected_progression = pool_sorted.iloc[idxs].drop_duplicates("triad_idx")
    if len(selected_progression) < N_PROGRESSION_ROWS:
        extra = pool_sorted[~pool_sorted["triad_idx"].isin(selected_progression["triad_idx"])]
        selected_progression = pd.concat(
            [selected_progression, extra.head(N_PROGRESSION_ROWS - len(selected_progression))],
            axis=0
        )

selected_realage = selected_realage.head(N_REALAGE_ROWS).sort_values("triad_idx").reset_index(drop=True)
selected_progression = selected_progression.head(N_PROGRESSION_ROWS).sort_values("triad_idx").reset_index(drop=True)

SEL_REALAGE_CSV = f"{OUT_DIR}/meta/real_sel.csv"
SEL_PROGRESS_CSV = f"{OUT_DIR}/meta/prog_sel.csv"

selected_realage.to_csv(SEL_REALAGE_CSV, index=False)
selected_progression.to_csv(SEL_PROGRESS_CSV, index=False)

print("Figura 1 CHILD-ONLY — selección:")
display(selected_realage[[
    "triad_idx", "real_age_child", "nearest_existing_child_age", "gender",
    "mean_S_id", "mean_S_family", "figure_score"
]])

print("\nFigura 2 CHILD-ONLY — selección:")
display(selected_progression[[
    "triad_idx", "real_age_child", "nearest_existing_child_age", "gender",
    "mean_S_id", "mean_S_family", "figure_score"
]])

print("Guardado:", SEL_REALAGE_CSV)
print("Guardado:", SEL_PROGRESS_CSV)

Figura 1 CHILD-ONLY — selección:


,triad_idx,real_age_child,nearest_existing_child_age,gender,mean_S_id,mean_S_family,figure_score
0,53,9,5,woman,0.322190,0.092529,0.850962
1,126,5,5,man,0.319355,0.040284,0.776442
2,127,2,5,man,0.232044,0.048482,0.734375
3,130,10,5,man,0.286577,0.028453,0.795673
4,149,12,15,man,0.242055,0.034242,0.740385
5,184,10,5,man,0.400207,0.030100,0.714663
6,186,6,5,woman,0.316862,0.051410,0.734856
7,203,12,15,woman,0.233171,0.044482,0.736779



Figura 2 CHILD-ONLY — selección:


,triad_idx,real_age_child,nearest_existing_child_age,gender,mean_S_id,mean_S_family,figure_score
0,53,9,5,woman,0.322190,0.092529,0.850962
1,55,8,5,woman,0.189285,0.040498,0.588221
2,174,5,5,woman,0.173068,0.031583,0.710337
3,202,8,5,man,0.247112,0.020009,0.619712
4,204,5,5,woman,0.190538,0.037333,0.666346


Guardado: /content/drive/MyDrive/Notebook4/results/figs_child/meta/real_sel.csv
Guardado: /content/drive/MyDrive/Notebook4/results/figs_child/meta/prog_sel.csv


## Bloque GPU opcional

Solo ejecútalo si quieres una progresión infantil más completa con edades:
`5, 8, 10, 12, 15`.

Si `GENERATE_CHILD_GRID = False`, la Figura 2 usará solo `Age 5` y `Age 15`, sin gastar GPU.

In [ ]:
# === Arranque GPU comun del proyecto ===
# Bootstrap Colab/Drive + src path
import os, sys

DRIVE = "/content/drive/MyDrive/Notebook4"
try:
    from google.colab import drive
    drive.mount("/content/drive")
except Exception:
    pass

for _path in [DRIVE, "/content", "/content/IP-Adapter"]:
    if _path not in sys.path:
        sys.path.insert(0, _path)

from src.notebook_setup import setup_data, setup_generation_models, setup_final_bap2
from src.inference_utils import CHILD_NEGATIVE_PROMPT, generate_bap2_image, get_embedding

ctx = setup_data(ctx) if getattr(ctx, 'data', None) is None else ctx
ctx = setup_generation_models(ctx) if getattr(ctx, 'pipe', None) is None else ctx
ctx = setup_final_bap2(ctx, parent_scale=0.1) if getattr(ctx, 'bap2_modules', None) is None else ctx

paths = ctx.paths
runtime = ctx.runtime
data = ctx.data
pipe = ctx.pipe
ip_model = ctx.ip_model
bap2_modules = ctx.bap2_modules

DEVICE = runtime.device
DTYPE = runtime.dtype
FIW_ROOT = str(paths.fiw_root)
embeddings = data.embeddings
gender_dict = data.gender_dict
test_ds = data.test_ds

GUIDANCE = globals().get('GUIDANCE', 4.0)
STEPS = globals().get('STEPS', 30)
NEG_PROMPT = CHILD_NEGATIVE_PROMPT

def get_emb(key):
    return get_embedding(embeddings, key)

@__import__('torch').inference_mode()
def generate_child_bap2(h_c, h_p, h_m, gender, age, seed, parent_scale=0.1):
    img, prompt, _alpha_stats = generate_bap2_image(
        pipe=pipe,
        ip_model=ip_model,
        h_c=h_c,
        h_p=h_p,
        h_m=h_m,
        gender=gender,
        age=age,
        device=DEVICE,
        dtype=DTYPE,
        parent_scale=parent_scale,
        seed=seed,
        guidance_scale=GUIDANCE,
        num_inference_steps=STEPS,
        negative_prompt=CHILD_NEGATIVE_PROMPT,
        child_prompt=True,
    )
    return img, prompt

print('Contexto GPU CHILD-ONLY listo')
print('BAP2 modules:', len(bap2_modules))


In [13]:
# ============================================================
# 11. Prueba mínima — generar UNA imagen infantil
# Antes de lanzar toda la progresión
# ============================================================

import os
import pandas as pd
from IPython.display import display

# ------------------------------------------------------------
# Recuperar selección si no está en memoria
# ------------------------------------------------------------
if "OUT_DIR" not in globals():
    OUT_DIR = f"{DRIVE}/results/figs_child"

if "selected_progression" not in globals():
    SEL_PROGRESS_CSV = f"{OUT_DIR}/meta/prog_sel.csv"
    print("Cargando selected_progression desde:", SEL_PROGRESS_CSV)
    assert os.path.exists(SEL_PROGRESS_CSV), SEL_PROGRESS_CSV
    selected_progression = pd.read_csv(SEL_PROGRESS_CSV)

print("selected_progression filas:", len(selected_progression))
display(selected_progression.head())

# ------------------------------------------------------------
# Tomar una tríada de prueba
# ------------------------------------------------------------
# ============================================================
# Repetir prueba mínima con prompt corregido
# ============================================================

row = selected_progression.iloc[0]

triad_idx = int(row["triad_idx"])
gender = row["gender"]
age_test = 8
seed_test = 12346  # cambiamos seed

h_c = get_emb(row["key_c"])
h_p = get_emb(row["key_p"])
h_m = get_emb(row["key_m"])

img, prompt = generate_child_bap2(
    h_c=h_c,
    h_p=h_p,
    h_m=h_m,
    gender=gender,
    age=age_test,
    seed=seed_test,
    parent_scale=0.1,
)

print(prompt)
display(img)

TEST_DIR = f"{OUT_DIR}/test"
os.makedirs(TEST_DIR, exist_ok=True)

test_path = f"{TEST_DIR}/triad_{triad_idx:04d}_age_{age_test}_test_prompt_fixed.jpg"
img.save(test_path, quality=95)

print("✓ Guardada en:", test_path)

selected_progression filas: 5


,triad_idx,n_ages,mean_S_id,mean_S_family,mean_gap,key_p,key_m,key_c,gender,real_age_child,...,real_sharpness_mean,real_contrast_mean,real_colorfulness_mean,has_gray_like_real,rank_id,rank_family,rank_sharp,rank_contrast,rank_color,figure_score
0,53,5,0.322190,0.092529,0.089790,F0294/MID2/P03124_face1.jpg,F0294/MID1/P03124_face0.jpg,F0294/MID3/P03124_face2.jpg,woman,9,...,0.015687,0.225262,0.140181,False,0.966346,0.995192,0.855769,0.706731,0.350962,0.850962
1,55,5,0.189285,0.040498,0.034429,F0309/MID2/P12240_face0.jpg,F0309/MID6/P12245_face1.jpg,F0309/MID4/P12243_face0.jpg,woman,8,...,0.009005,0.199886,0.181894,False,0.514423,0.793269,0.538462,0.394231,0.687500,0.588221
2,174,5,0.173068,0.031583,0.038879,F0965/MID1/P10163_face2.jpg,F0965/MID4/P10163_face0.jpg,F0965/MID5/P10163_face1.jpg,woman,5,...,0.033150,0.247232,0.210429,False,0.447115,0.682692,0.971154,0.822115,0.879808,0.710337
3,202,5,0.247112,0.020009,0.095335,F1005/MID1/P13041_face0.jpg,F1005/MID2/P13041_face1.jpg,F1005/MID4/P13041_face3.jpg,man,8,...,0.014593,0.202967,0.138636,False,0.788462,0.495192,0.812500,0.427885,0.326923,0.619712
4,204,5,0.190538,0.037333,0.084788,F1007/MID1/P13062_face4.jpg,F1007/MID2/P13062_face2.jpg,F1007/MID4/P13062_face6.jpg,woman,5,...,0.009613,0.280634,0.173745,False,0.524038,0.740385,0.581731,0.961538,0.634615,0.666346


Token indices sequence length is longer than the specified maximum sequence length for this model (82 > 77). Running this sequence through the model will result in indexing errors


  0%|          | 0/30 [00:00<?, ?it/s]

realistic portrait of a girl, age 8, full face visible, centered, head and shoulders, looking at camera, neutral expression, natural appearance, soft natural lighting, visible hair, visible forehead, visible chin, simple background


<PIL.Image.Image image mode=RGB size=512x512>

✓ Guardada en: /content/drive/MyDrive/Notebook4/results/figs_child/test/t0053_a08_test.jpg


In [24]:
# ============================================================
# 12. GPU opcional — generar imagen a edad real infantil exacta
# ============================================================

if GENERATE_REAL_CHILD_AGE_EXACT:
    rows = []

    for _, row in selected_realage.iterrows():
        triad_idx = int(row["triad_idx"])
        age_real = float(row["real_age_child"])
        age_int = int(round(age_real))
        age_int = max(1, min(age_int, MAX_CHILD_REAL_AGE))

        gender = row["gender"] if row["gender"] in ["man", "woman", "person"] else "person"

        out_path = f"{CHILD_GEN_DIR}/triad_{triad_idx:04d}_real_child_age_{age_int}_bap2.jpg"

        if os.path.exists(out_path):
            print("skip existe:", out_path)
        else:
            h_c = get_emb(row["key_c"])
            h_p = get_emb(row["key_p"])
            h_m = get_emb(row["key_m"])

            seed = 920000 + triad_idx * 100 + age_int

            img, prompt = generate_child_bap2(
                h_c=h_c,
                h_p=h_p,
                h_m=h_m,
                gender=gender,
                age=age_int,
                seed=seed,
                parent_scale=0.1,
            )

            img.save(out_path, quality=95)
            print("✓ generado:", out_path, "|", prompt)

        rows.append({
            "triad_idx": triad_idx,
            "real_age_child": age_real,
            "rounded_age": age_int,
            "generated_path": out_path,
        })

    real_child_age_df = pd.DataFrame(rows)
    real_child_age_csv = f"{OUT_DIR}/meta/real_age.csv"
    real_child_age_df.to_csv(real_child_age_csv, index=False)

    print("✓ CSV:", real_child_age_csv)
    display(real_child_age_df)
else:
    print("Saltado: GENERATE_REAL_CHILD_AGE_EXACT=False")

  0%|          | 0/30 [00:00<?, ?it/s]

✓ generado: /content/drive/MyDrive/Notebook4/results/figs_child/grid/t0053_a09.jpg | realistic portrait of a girl, age 9, full face visible, centered, head and shoulders, looking at camera, neutral expression, natural appearance, soft natural lighting, visible hair, visible forehead, visible chin, simple background


  0%|          | 0/30 [00:00<?, ?it/s]

✓ generado: /content/drive/MyDrive/Notebook4/results/figs_child/grid/t0126_a05.jpg | realistic portrait of a boy, age 5, full face visible, centered, head and shoulders, looking at camera, neutral expression, natural appearance, soft natural lighting, visible hair, visible forehead, visible chin, simple background


  0%|          | 0/30 [00:00<?, ?it/s]

...

  0%|          | 0/30 [00:00<?, ?it/s]

✓ generado: /content/drive/MyDrive/Notebook4/results/figs_child/grid/t0186_a06.jpg | realistic portrait of a girl, age 6, full face visible, centered, head and shoulders, looking at camera, neutral expression, natural appearance, soft natural lighting, visible hair, visible forehead, visible chin, simple background


  0%|          | 0/30 [00:00<?, ?it/s]

✓ generado: /content/drive/MyDrive/Notebook4/results/figs_child/grid/t0203_a12.jpg | realistic portrait of a girl, age 12, full face visible, centered, head and shoulders, looking at camera, neutral expression, natural appearance, soft natural lighting, visible hair, visible forehead, visible chin, simple background
✓ CSV: /content/drive/MyDrive/Notebook4/results/figs_child/meta/real_age.csv


,triad_idx,real_age_child,rounded_age,generated_path
0,53,9.0,9,/content/drive/MyDrive/Notebook4/results/final...
1,126,5.0,5,/content/drive/MyDrive/Notebook4/results/final...
2,127,2.0,2,/content/drive/MyDrive/Notebook4/results/final...
3,130,10.0,10,/content/drive/MyDrive/Notebook4/results/final...
4,149,12.0,12,/content/drive/MyDrive/Notebook4/results/final...
5,184,10.0,10,/content/drive/MyDrive/Notebook4/results/final...
6,186,6.0,6,/content/drive/MyDrive/Notebook4/results/final...
7,203,12.0,12,/content/drive/MyDrive/Notebook4/results/final...


In [25]:
# ============================================================
# 13. Preparar rutas para Figura 1 CHILD-ONLY
# ============================================================

selected_realage_fig = selected_realage.copy()

if GENERATE_REAL_CHILD_AGE_EXACT and os.path.exists(f"{OUT_DIR}/meta/real_age.csv"):
    real_child_age_df = pd.read_csv(f"{OUT_DIR}/meta/real_age.csv")
    selected_realage_fig = selected_realage_fig.merge(
        real_child_age_df[["triad_idx", "rounded_age", "generated_path"]],
        on="triad_idx",
        how="left"
    )
    selected_realage_fig["fig_generated_path"] = selected_realage_fig["generated_path"]
    selected_realage_fig["fig_generated_age"] = selected_realage_fig["rounded_age"]
    selected_realage_fig["fig_age_mode"] = "exact_real_child_age"
else:
    selected_realage_fig["fig_generated_age"] = selected_realage_fig["nearest_existing_child_age"].astype(int)
    selected_realage_fig["fig_generated_path"] = selected_realage_fig.apply(
        lambda r: get_existing_bap2_image_path(r["triad_idx"], r["nearest_existing_child_age"]),
        axis=1
    )
    selected_realage_fig["fig_age_mode"] = "nearest_existing_child_age"

display(selected_realage_fig[[
    "triad_idx", "real_age_child", "nearest_existing_child_age",
    "fig_generated_age", "fig_age_mode", "fig_generated_path"
]])

selected_realage_fig.to_csv(f"{OUT_DIR}/meta/fig1_paths.csv", index=False)

,triad_idx,real_age_child,nearest_existing_child_age,fig_generated_age,fig_age_mode,fig_generated_path
0,53,9,5,9,exact_real_child_age,/content/drive/MyDrive/Notebook4/results/final...
1,126,5,5,5,exact_real_child_age,/content/drive/MyDrive/Notebook4/results/final...
2,127,2,5,2,exact_real_child_age,/content/drive/MyDrive/Notebook4/results/final...
3,130,10,5,10,exact_real_child_age,/content/drive/MyDrive/Notebook4/results/final...
4,149,12,15,12,exact_real_child_age,/content/drive/MyDrive/Notebook4/results/final...
5,184,10,5,10,exact_real_child_age,/content/drive/MyDrive/Notebook4/results/final...
6,186,6,5,6,exact_real_child_age,/content/drive/MyDrive/Notebook4/results/final...
7,203,12,15,12,exact_real_child_age,/content/drive/MyDrive/Notebook4/results/final...


In [26]:
# ============================================================
# 14. FIGURA 1 CHILD-ONLY
# Father | Mother | Real Child | Ours
# ============================================================

columns = ["Father", "Mother", "Real Child", "Ours"]

n_rows = len(selected_realage_fig)
n_cols = len(columns)

fig_w = n_cols * 2.3
fig_h = n_rows * 2.25 + 0.8

fig, axes = plt.subplots(n_rows, n_cols, figsize=(fig_w, fig_h), dpi=DPI)

if n_rows == 1:
    axes = np.expand_dims(axes, axis=0)

for r, row in selected_realage_fig.iterrows():
    paths = [
        key_to_path(row["key_p"]),
        key_to_path(row["key_m"]),
        key_to_path(row["key_c"]),
        row["fig_generated_path"],
    ]

    for c, path in enumerate(paths):
        ax = axes[r, c]
        img = prep_img(path, size=IMG_SIZE)
        ax.imshow(img)
        ax.axis("off")

        if r == 0:
            if c == 3:
                ax.set_title("Ours\n(child age)", fontsize=10, pad=6)
            else:
                ax.set_title(columns[c], fontsize=10, pad=6)

    label = (
        f"T{int(row['triad_idx'])}\n"
        f"real={row['real_age_child']:.1f}\n"
        f"gen={int(row['fig_generated_age'])}"
    )
    axes[r, 0].text(
        -0.22, 0.5,
        label,
        transform=axes[r, 0].transAxes,
        rotation=90,
        va="center",
        ha="center",
        fontsize=7
    )

plt.subplots_adjust(left=0.06, right=0.995, top=0.92, bottom=0.03, wspace=0.035, hspace=0.06)

title = "Child-only qualitative comparison"
if not GENERATE_REAL_CHILD_AGE_EXACT:
    title += " (nearest generated child age)"

fig.suptitle(title, fontsize=12, y=0.985)

FIG1_PNG = f"{OUT_DIR}/final/fig1_real.png"

save_fig(fig, FIG1_PNG, FIG1_PDF)
plt.show()

✓ Guardado: /content/drive/MyDrive/Notebook4/results/figs_child/final/fig1_real.png


<Figure size 2760x5640 with 32 Axes>

In [27]:
# ============================================================
# Crear hoja de candidatos CHILD-ONLY para selección manual
# Father | Mother | Real Child | Ours nearest child age
# ============================================================

import os
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from IPython.display import display

# ------------------------------------------------------------
# Configuración
# ------------------------------------------------------------

N_CANDIDATES = 30
CANDIDATE_COLS = ["Father", "Mother", "Real Child", "Ours"]

if "OUT_DIR" not in globals():
    OUT_DIR = f"{DRIVE}/results/figs_child"

os.makedirs(OUT_DIR, exist_ok=True)

# ------------------------------------------------------------
# Asegurar pool infantil
# ------------------------------------------------------------

assert "vis" in globals(), "Falta vis. Corre la celda de score visual CHILD-ONLY."
assert "bap2_on" in globals(), "Falta bap2_on."
assert "get_existing_bap2_image_path" in globals(), "Falta get_existing_bap2_image_path."
assert "prep_img" in globals(), "Falta prep_img."
assert "key_to_path" in globals(), "Falta key_to_path."

pool = vis[
    (vis["real_age_child"].notna()) &
    (vis["real_age_child"] >= 0) &
    (vis["real_age_child"] <= MAX_CHILD_REAL_AGE)
].copy()

# quitar imágenes reales demasiado malas
pool = pool[
    (pool["real_min_side"] >= 90) &
    (pool["real_sharpness_mean"] > pool["real_sharpness_mean"].quantile(0.30))
].copy()

# si existe columna de grayscale, evitar blanco y negro
if "has_gray_like_real" in pool.columns:
    pool = pool[~pool["has_gray_like_real"]].copy()

# evitar padres repetidos exactos
pool["parent_pair"] = pool["key_p"].astype(str) + "||" + pool["key_m"].astype(str)
pool = pool.sort_values("figure_score", ascending=False)
pool = pool.drop_duplicates(subset=["parent_pair"], keep="first")

# seleccionar top candidatos
candidates = pool.head(N_CANDIDATES).copy().reset_index(drop=True)

# edad generada más cercana infantil
if "nearest_existing_child_age" not in candidates.columns:
    candidates["nearest_existing_child_age"] = candidates["real_age_child"].apply(
        lambda x: 5 if abs(float(x)-5) <= abs(float(x)-15) else 15
    )

candidates["ours_path"] = candidates.apply(
    lambda r: get_existing_bap2_image_path(r["triad_idx"], r["nearest_existing_child_age"]),
    axis=1
)

CANDIDATES_CSV = f"{OUT_DIR}/meta/candidates.csv"
candidates.to_csv(CANDIDATES_CSV, index=False)

print("Candidatos guardados en:", CANDIDATES_CSV)
display(candidates[[
    "triad_idx",
    "real_age_child",
    "nearest_existing_child_age",
    "gender",
    "mean_S_id",
    "mean_S_family",
    "real_min_side",
    "real_sharpness_mean",
    "figure_score",
]])

# ------------------------------------------------------------
# Dibujar hoja de candidatos
# ------------------------------------------------------------

n_rows = len(candidates)
n_cols = 4

fig_w = n_cols * 2.2
fig_h = n_rows * 1.95 + 0.8

fig, axes = plt.subplots(n_rows, n_cols, figsize=(fig_w, fig_h), dpi=180)

if n_rows == 1:
    axes = np.expand_dims(axes, axis=0)

for r, row in candidates.iterrows():
    paths = [
        key_to_path(row["key_p"]),
        key_to_path(row["key_m"]),
        key_to_path(row["key_c"]),
        row["ours_path"],
    ]

    for c, path in enumerate(paths):
        ax = axes[r, c]
        img = prep_img(path, size=210)
        ax.imshow(img)
        ax.axis("off")

        if r == 0:
            ax.set_title(CANDIDATE_COLS[c], fontsize=9, pad=5)

    label = (
        f"T{int(row['triad_idx'])}\n"
        f"real={row['real_age_child']:.1f}\n"
        f"gen={int(row['nearest_existing_child_age'])}"
    )

    axes[r, 0].text(
        -0.24, 0.5,
        label,
        transform=axes[r, 0].transAxes,
        rotation=90,
        va="center",
        ha="center",
        fontsize=7
    )

plt.subplots_adjust(
    left=0.07,
    right=0.995,
    top=0.965,
    bottom=0.02,
    wspace=0.035,
    hspace=0.06
)

fig.suptitle("Child-only candidate sheet for manual qualitative selection", fontsize=11, y=0.995)

CANDIDATES_PNG = f"{OUT_DIR}/final/candidates.png"
fig.savefig(CANDIDATES_PNG, dpi=300, bbox_inches="tight", facecolor="white")
plt.show()

print("✓ Hoja guardada en:", CANDIDATES_PNG)

Candidatos guardados en: /content/drive/MyDrive/Notebook4/results/figs_child/meta/candidates.csv


,triad_idx,real_age_child,nearest_existing_child_age,gender,mean_S_id,mean_S_family,real_min_side,real_sharpness_mean,figure_score
0,53,9,5,woman,0.322190,0.092529,108,0.015687,0.850962
1,130,10,5,man,0.286577,0.028453,108,0.013433,0.795673
2,126,5,5,man,0.319355,0.040284,108,0.005957,0.776442
3,203,12,15,woman,0.233171,0.044482,108,0.008953,0.736779
4,186,6,5,woman,0.316862,0.051410,108,0.010751,0.734856
5,174,5,5,woman,0.173068,0.031583,108,0.033150,0.710337
6,109,2,5,woman,0.212748,0.009321,108,0.027667,0.705529
7,112,12,15,woman,0.194791,0.056533,108,0.011637,0.694231
8,80,10,5,man,0.275519,0.014429,108,0.059397,0.671875
9,63,6,5,man,0.250672,0.003392,108,0.017784,0.658894


<Figure size 1584x10674 with 120 Axes>

✓ Hoja guardada en: /content/drive/MyDrive/Notebook4/results/figs_child/final/candidates.png


In [28]:
# ============================================================
# Figura final CHILD-ONLY elegida manualmente
# Father | Mother | Real Child | Ours
# ============================================================

import os
import re
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

# ------------------------------------------------------------
# Configuración
# ------------------------------------------------------------
if "DRIVE" not in globals():
    DRIVE = "/content/drive/MyDrive/Notebook4"

if "OUT_DIR" not in globals():
    OUT_DIR = f"{DRIVE}/results/figs_child"

os.makedirs(OUT_DIR, exist_ok=True)

CANDIDATES_CSV = f"{OUT_DIR}/meta/candidates.csv"
assert os.path.exists(CANDIDATES_CSV), f"No existe {CANDIDATES_CSV}"

# Tus IDs manuales
MANUAL_IDS_RAW = [166, "T174", 109, 71, 47, 173, 202, 150, 164, 52]

# ------------------------------------------------------------
# Helpers
# ------------------------------------------------------------
def normalize_triad_id(x):
    s = str(x).strip().upper()
    s = s.replace("T", "")
    return int(s)

manual_ids = [normalize_triad_id(x) for x in MANUAL_IDS_RAW]
print("Triadas manuales:", manual_ids)

# ------------------------------------------------------------
# Cargar candidatos y filtrar
# ------------------------------------------------------------
candidates = pd.read_csv(CANDIDATES_CSV)

assert "triad_idx" in candidates.columns, "meta/candidates.csv no tiene triad_idx"

selected_final = candidates[candidates["triad_idx"].isin(manual_ids)].copy()

# mantener el orden exacto que tú diste
selected_final["manual_order"] = selected_final["triad_idx"].apply(lambda x: manual_ids.index(int(x)))
selected_final = selected_final.sort_values("manual_order").reset_index(drop=True)

print("Filas seleccionadas:", len(selected_final))
print("Triadas encontradas:", selected_final["triad_idx"].tolist())

missing = [tid for tid in manual_ids if tid not in selected_final["triad_idx"].tolist()]
if len(missing) > 0:
    print("⚠ Faltaron estas triadas en el CSV de candidatos:", missing)

display(selected_final[[
    "triad_idx",
    "real_age_child",
    "nearest_existing_child_age",
    "gender",
    "mean_S_id",
    "mean_S_family",
    "figure_score"
]])

# ------------------------------------------------------------
# Verificaciones mínimas
# ------------------------------------------------------------
assert "prep_img" in globals(), "Falta prep_img. Corre las celdas de helpers de imagen."
assert "key_to_path" in globals(), "Falta key_to_path. Corre las celdas de helpers de imagen."
assert "get_existing_bap2_image_path" in globals(), "Falta get_existing_bap2_image_path."

# ------------------------------------------------------------
# Resolver path de imagen generada
# ------------------------------------------------------------
if "ours_path" not in selected_final.columns:
    selected_final["ours_path"] = selected_final.apply(
        lambda r: get_existing_bap2_image_path(r["triad_idx"], r["nearest_existing_child_age"]),
        axis=1
    )

# Guardar selección final
FINAL_SEL_CSV = f"{OUT_DIR}/meta/fig1_sel.csv"
selected_final.to_csv(FINAL_SEL_CSV, index=False)
print("✓ Selección guardada en:", FINAL_SEL_CSV)

# ------------------------------------------------------------
# Dibujar figura final
# ------------------------------------------------------------
columns = ["Father", "Mother", "Real Child", "Ours"]

n_rows = len(selected_final)
n_cols = len(columns)

fig_w = n_cols * 2.35
fig_h = n_rows * 2.10 + 0.8

fig, axes = plt.subplots(n_rows, n_cols, figsize=(fig_w, fig_h), dpi=300)

if n_rows == 1:
    axes = np.expand_dims(axes, axis=0)

for r, row in selected_final.iterrows():
    paths = [
        key_to_path(row["key_p"]),
        key_to_path(row["key_m"]),
        key_to_path(row["key_c"]),
        row["ours_path"],
    ]

    for c, path in enumerate(paths):
        ax = axes[r, c]
        img = prep_img(path, size=220)
        ax.imshow(img)
        ax.axis("off")

        if r == 0:
            if c == 3:
                ax.set_title("Ours", fontsize=10, pad=6)
            else:
                ax.set_title(columns[c], fontsize=10, pad=6)

    label = (
        f"T{int(row['triad_idx'])}\n"
        f"real={row['real_age_child']:.1f}\n"
        f"gen={int(row['nearest_existing_child_age'])}"
    )
    axes[r, 0].text(
        -0.22, 0.5,
        label,
        transform=axes[r, 0].transAxes,
        rotation=90,
        va="center",
        ha="center",
        fontsize=7
    )

plt.subplots_adjust(left=0.07, right=0.995, top=0.955, bottom=0.02, wspace=0.04, hspace=0.06)
fig.suptitle("Child-only qualitative comparison", fontsize=12, y=0.99)

FIG1_PNG = f"{OUT_DIR}/final/fig1.png"

fig.savefig(FIG1_PNG, dpi=300, bbox_inches="tight", facecolor="white")
fig.savefig(FIG1_PDF, bbox_inches="tight", facecolor="white")

print("✓ Figura PNG:", FIG1_PNG)

plt.show()

Triadas manuales: [166, 174, 109, 71, 47, 173, 202, 150, 164, 52]
Filas seleccionadas: 10
Triadas encontradas: [166, 174, 109, 71, 47, 173, 202, 150, 164, 52]


,triad_idx,real_age_child,nearest_existing_child_age,gender,mean_S_id,mean_S_family,figure_score
0,166,4,5,man,0.175683,0.006127,0.562500
1,174,5,5,woman,0.173068,0.031583,0.710337
2,109,2,5,woman,0.212748,0.009321,0.705529
3,71,11,15,woman,0.162674,0.021940,0.604808
4,47,6,5,woman,0.253903,0.013554,0.528606
5,173,4,5,man,0.071189,-0.027872,0.356971
6,202,8,5,man,0.247112,0.020009,0.619712
7,150,7,5,man,0.166255,0.023555,0.627885
8,164,2,5,man,0.253277,0.023036,0.631490
9,52,1,5,woman,0.190549,0.000314,0.524519


✓ Selección guardada en: /content/drive/MyDrive/Notebook4/results/figs_child/meta/fig1_sel.csv
✓ Figura PNG: /content/drive/MyDrive/Notebook4/results/figs_child/final/fig1.png


<Figure size 2820x6540 with 40 Axes>

In [29]:
# ============================================================
# Figura 2 — generar progresión infantil/manual
# Triadas fijas + edades fijas
# ============================================================

import os
import pandas as pd
from IPython.display import display

# ------------------------------------------------------------
# Configuración
# ------------------------------------------------------------
if "DRIVE" not in globals():
    DRIVE = "/content/drive/MyDrive/Notebook4"

if "OUT_DIR" not in globals():
    OUT_DIR = f"{DRIVE}/results/figs_child"

os.makedirs(OUT_DIR, exist_ok=True)

PROGRESSION_DIR = f"{OUT_DIR}/prog"
os.makedirs(PROGRESSION_DIR, exist_ok=True)

MANUAL_TRIADS = [166, 174, 109, 71, 202, 173]
PROG_AGES = [5, 10, 15, 20, 30, 40]

# ------------------------------------------------------------
# Cargar selección base
# ------------------------------------------------------------
SEL_CSV_1 = f"{OUT_DIR}/meta/fig1_sel.csv"
SEL_CSV_2 = f"{OUT_DIR}/meta/candidates.csv"

if os.path.exists(SEL_CSV_1):
    base_df = pd.read_csv(SEL_CSV_1)
    print("✓ Cargado:", SEL_CSV_1)
elif os.path.exists(SEL_CSV_2):
    base_df = pd.read_csv(SEL_CSV_2)
    print("✓ Cargado:", SEL_CSV_2)
else:
    raise FileNotFoundError("No encontré CSV base de selección.")

prog_df = base_df[base_df["triad_idx"].isin(MANUAL_TRIADS)].copy()

# mantener orden exacto
prog_df["manual_order"] = prog_df["triad_idx"].apply(lambda x: MANUAL_TRIADS.index(int(x)))
prog_df = prog_df.sort_values("manual_order").reset_index(drop=True)

print("Triadas encontradas:", prog_df["triad_idx"].tolist())
display(prog_df[["triad_idx", "real_age_child", "gender", "key_p", "key_m", "key_c"]])

# ------------------------------------------------------------
# Verificaciones
# ------------------------------------------------------------
assert "generate_child_bap2" in globals(), "Falta generate_child_bap2"
assert "get_emb" in globals(), "Falta get_emb"

# ------------------------------------------------------------
# Generar imágenes
# ------------------------------------------------------------
rows = []

for _, row in prog_df.iterrows():
    triad_idx = int(row["triad_idx"])
    gender = row["gender"] if row["gender"] in ["man", "woman", "person"] else "person"

    h_c = get_emb(row["key_c"])
    h_p = get_emb(row["key_p"])
    h_m = get_emb(row["key_m"])

    print(f"\nTriada {triad_idx}")

    for age in PROG_AGES:
        out_path = f"{PROGRESSION_DIR}/triad_{triad_idx:04d}_age_{age:02d}_bap2.jpg"

        if os.path.exists(out_path):
            print("skip existe:", os.path.basename(out_path))
        else:
            seed = 940000 + triad_idx * 100 + age

            img, prompt = generate_child_bap2(
                h_c=h_c,
                h_p=h_p,
                h_m=h_m,
                gender=gender,
                age=age,
                seed=seed,
                parent_scale=0.1,
            )

            img.save(out_path, quality=95)
            print("✓ generado:", os.path.basename(out_path), "|", prompt)

        rows.append({
            "triad_idx": triad_idx,
            "gender": gender,
            "real_age_child": row["real_age_child"] if "real_age_child" in row else None,
            "key_p": row["key_p"],
            "key_m": row["key_m"],
            "key_c": row["key_c"],
            "age": age,
            "image_path": out_path,
        })

progression_paths_df = pd.DataFrame(rows)
PROG_CSV = f"{OUT_DIR}/meta/fig2_paths.csv"
progression_paths_df.to_csv(PROG_CSV, index=False)

print("\n✓ CSV guardado en:", PROG_CSV)
display(progression_paths_df.head(12))

✓ Cargado: /content/drive/MyDrive/Notebook4/results/figs_child/meta/fig1_sel.csv
Triadas encontradas: [166, 174, 109, 71, 202, 173]


,triad_idx,real_age_child,gender,key_p,key_m,key_c
0,166,4,man,F0952/MID1/P10025_face0.jpg,F0952/MID3/P10026_face3.jpg,F0952/MID2/P10025_face1.jpg
1,174,5,woman,F0965/MID1/P10163_face2.jpg,F0965/MID4/P10163_face0.jpg,F0965/MID5/P10163_face1.jpg
2,109,2,woman,F0637/MID1/P06692_face2.jpg,F0637/MID2/P06692_face1.jpg,F0637/MID3/P06692_face0.jpg
3,71,11,woman,F0427/MID5/P04490_face2.jpg,F0427/MID1/P04490_face0.jpg,F0427/MID3/P04490_face1.jpg
4,202,8,man,F1005/MID1/P13041_face0.jpg,F1005/MID2/P13041_face1.jpg,F1005/MID4/P13041_face3.jpg
5,173,4,man,F0965/MID1/P10163_face2.jpg,F0965/MID2/P10164_face3.jpg,F0965/MID3/P10166_face1.jpg



Triada 166


  0%|          | 0/30 [00:00<?, ?it/s]

✓ generado: t0166_a05.jpg | realistic portrait of a boy, age 5, full face visible, centered, head and shoulders, looking at camera, neutral expression, natural appearance, soft natural lighting, visible hair, visible forehead, visible chin, simple background


...

  0%|          | 0/30 [00:00<?, ?it/s]

✓ generado: t0173_a30.jpg | realistic portrait of a teenage boy, age 30, full face visible, centered, head and shoulders, looking at camera, neutral expression, natural appearance, soft natural lighting, visible hair, visible forehead, visible chin, simple background


  0%|          | 0/30 [00:00<?, ?it/s]

✓ generado: t0173_a40.jpg | realistic portrait of a teenage boy, age 40, full face visible, centered, head and shoulders, looking at camera, neutral expression, natural appearance, soft natural lighting, visible hair, visible forehead, visible chin, simple background

✓ CSV guardado en: /content/drive/MyDrive/Notebook4/results/figs_child/meta/fig2_paths.csv


,triad_idx,gender,real_age_child,key_p,key_m,key_c,age,image_path
0,166,man,4,F0952/MID1/P10025_face0.jpg,F0952/MID3/P10026_face3.jpg,F0952/MID2/P10025_face1.jpg,5,/content/drive/MyDrive/Notebook4/results/final...
1,166,man,4,F0952/MID1/P10025_face0.jpg,F0952/MID3/P10026_face3.jpg,F0952/MID2/P10025_face1.jpg,10,/content/drive/MyDrive/Notebook4/results/final...
2,166,man,4,F0952/MID1/P10025_face0.jpg,F0952/MID3/P10026_face3.jpg,F0952/MID2/P10025_face1.jpg,15,/content/drive/MyDrive/Notebook4/results/final...
3,166,man,4,F0952/MID1/P10025_face0.jpg,F0952/MID3/P10026_face3.jpg,F0952/MID2/P10025_face1.jpg,20,/content/drive/MyDrive/Notebook4/results/final...
4,166,man,4,F0952/MID1/P10025_face0.jpg,F0952/MID3/P10026_face3.jpg,F0952/MID2/P10025_face1.jpg,30,/content/drive/MyDrive/Notebook4/results/final...
5,166,man,4,F0952/MID1/P10025_face0.jpg,F0952/MID3/P10026_face3.jpg,F0952/MID2/P10025_face1.jpg,40,/content/drive/MyDrive/Notebook4/results/final...
6,174,woman,5,F0965/MID1/P10163_face2.jpg,F0965/MID4/P10163_face0.jpg,F0965/MID5/P10163_face1.jpg,5,/content/drive/MyDrive/Notebook4/results/final...
7,174,woman,5,F0965/MID1/P10163_face2.jpg,F0965/MID4/P10163_face0.jpg,F0965/MID5/P10163_face1.jpg,10,/content/drive/MyDrive/Notebook4/results/final...
8,174,woman,5,F0965/MID1/P10163_face2.jpg,F0965/MID4/P10163_face0.jpg,F0965/MID5/P10163_face1.jpg,15,/content/drive/MyDrive/Notebook4/results/final...
9,174,woman,5,F0965/MID1/P10163_face2.jpg,F0965/MID4/P10163_face0.jpg,F0965/MID5/P10163_face1.jpg,20,/content/drive/MyDrive/Notebook4/results/final...


In [ ]:
# ============================================================
# 17. Figura 3 opcional — Limitaciones infantiles
# ============================================================

df_lim = bap2_on[bap2_on["age"].isin(EXISTING_CHILD_AGES)].copy()

# Unir edad real para mantener casos infantiles reales.
age_by_triad = triad_summary[["triad_idx", "real_age_child"]].drop_duplicates()
df_lim = df_lim.merge(age_by_triad, on="triad_idx", how="left")
df_lim = df_lim[
    (df_lim["real_age_child"].notna()) &
    (df_lim["real_age_child"] <= MAX_CHILD_REAL_AGE)
].copy()

df_lim["rank_low_id"] = 1.0 - df_lim["S_id_child"].rank(pct=True)
df_lim["rank_low_family"] = 1.0 - df_lim["S_family"].rank(pct=True)
df_lim["rank_high_gap"] = df_lim["balance_abs_gap"].rank(pct=True)

df_lim["difficulty_score"] = (
    0.45 * df_lim["rank_low_id"] +
    0.35 * df_lim["rank_low_family"] +
    0.20 * df_lim["rank_high_gap"]
)

selected_lim_rows = []
used = set()

for age in EXISTING_CHILD_AGES:
    sub = df_lim[(df_lim["age"] == age) & (~df_lim["triad_idx"].isin(used))]
    if len(sub) == 0:
        continue
    row = sub.sort_values("difficulty_score", ascending=False).iloc[0]
    selected_lim_rows.append(row)
    used.add(int(row["triad_idx"]))
    if len(selected_lim_rows) >= N_LIMITATION_ROWS:
        break

# completar si faltan
if len(selected_lim_rows) < N_LIMITATION_ROWS:
    extra = df_lim[~df_lim["triad_idx"].isin(used)].sort_values("difficulty_score", ascending=False)
    for _, row in extra.iterrows():
        selected_lim_rows.append(row)
        used.add(int(row["triad_idx"]))
        if len(selected_lim_rows) >= N_LIMITATION_ROWS:
            break

selected_lim = pd.DataFrame(selected_lim_rows).reset_index(drop=True)

display(selected_lim[[
    "triad_idx", "real_age_child", "age", "gender", "S_id_child", "S_family", "balance_abs_gap", "image_path"
]])

selected_lim.to_csv(f"{OUT_DIR}/figure3_child_limitations_selection.csv", index=False)

In [ ]:
# ============================================================
# 18. Dibujar Figura 3 opcional — Limitaciones infantiles
# Father | Mother | Real Child | Ours | Métricas
# ============================================================

columns = ["Father", "Mother", "Real Child", "Ours", "Metrics"]

n_rows = len(selected_lim)
n_cols = len(columns)

fig_w = 11.5
fig_h = n_rows * 2.1 + 0.8

fig, axes = plt.subplots(
    n_rows,
    n_cols,
    figsize=(fig_w, fig_h),
    dpi=DPI,
    gridspec_kw={"width_ratios": [1, 1, 1, 1, 1.6]}
)

if n_rows == 1:
    axes = np.expand_dims(axes, axis=0)

for r, row in selected_lim.iterrows():
    paths = [
        key_to_path(row["key_p"]),
        key_to_path(row["key_m"]),
        key_to_path(row["key_c"]),
        row["image_path"],
    ]

    for c, path in enumerate(paths):
        ax = axes[r, c]
        img = prep_img(path, size=IMG_SIZE)
        ax.imshow(img)
        ax.axis("off")

        if r == 0:
            ax.set_title(columns[c], fontsize=10, pad=6)

    ax = axes[r, 4]
    ax.axis("off")
    if r == 0:
        ax.set_title("Metrics", fontsize=10, pad=6)

    text = (
        f"T{int(row['triad_idx'])}, gen age={int(row['age'])}\n"
        f"real age={row['real_age_child']:.1f}\n"
        f"S_id={row['S_id_child']:.3f}\n"
        f"S_family={row['S_family']:.3f}\n"
        f"gap={row['balance_abs_gap']:.3f}"
    )

    ax.text(0.02, 0.5, text, va="center", ha="left", fontsize=8, wrap=True)

plt.subplots_adjust(left=0.035, right=0.995, top=0.91, bottom=0.03, wspace=0.05, hspace=0.08)

fig.suptitle("Child-only representative limitations", fontsize=12, y=0.985)

FIG3_PNG = f"{OUT_DIR}/figure3_child_only_limitations.png"

save_fig(fig, FIG3_PNG, FIG3_PDF)
plt.show()

In [ ]:
# ============================================================
# 19. Resumen de archivos generados
# ============================================================

print("Archivos generados en:", OUT_DIR)
for p in sorted(glob.glob(f"{OUT_DIR}/*")):
    print(os.path.basename(p), "|", p)

## Pies de figura sugeridos

**Figura 1.** Comparación cualitativa infantil entre las imágenes reales de la tríada familiar y la imagen generada por BAP2. Solo se seleccionaron casos donde la edad estimada del hijo real es menor o igual a 16 años.

**Figura 2.** Progresión infantil generada con BAP2. En el modo sin GPU se muestran las edades infantiles disponibles en la evaluación final: 5 y 15 años. En el modo con GPU se genera una secuencia más densa entre 5 y 15 años.

**Figura 3.** Casos infantiles representativos de limitaciones cualitativas, seleccionados por bajo desempeño biométrico o mayor desbalance parental.